# RQ1 v2.1 — GraphSAGE-Norm vs GraphSAGE+ — 20 split mới đã khóa trước

Đây là vòng xác nhận **v2.1**, được khóa sau khi đã xem toàn bộ kết quả v1 và trước khi chạy bất kỳ kết quả v2/v2.1 nào.

- Chạy đủ 20 split mới `2001–2020`; không dừng sớm theo p-value.
- Mỗi split có 1657/553/553 node và 5 model seed độc lập `12–16`.
- Hai biến thể dùng cùng split, cùng seed và cùng batch đã materialize. `GraphSAGE+` chỉ thêm 12 UFE label-free.
- Kiểm định chính: Nadeau–Bengio trên 20 delta cấp split; Holm được áp dụng sau khi đủ 6 cặp.
- Đây là kiểm tra độ ổn định trên cùng 2763 node, **không phải** external/temporal holdout và **không thay thế** Bảng A so với paper.

Master protocol SHA-256: `3e63674a98e052d1770851009078adf4e2ae192a47dc9c72b84c987d202aa17c`.


In [1]:

from pathlib import Path
import csv
import gc
import hashlib
import json
import os
import random
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone

import numpy as np
import torch

MODEL_KEY = 'graphsage'
MODEL_LABEL = 'GraphSAGE'
VARIANTS = ["base", "plus"]
VARIANT_LABEL = {"base": 'GraphSAGE-Norm', "plus": 'GraphSAGE+'}
SHARED_CONFIG = {'hidden': 32, 'learning_rate': 0.01}
MODEL_SEEDS = [12, 13, 14, 15, 16]
RESPLIT_SEEDS = [2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
EXPECTED_COUNTS = {"train": 1657, "validation": 553, "test": 553}
SOURCE_DATA_SHA256 = "8cc653bb389ea1a316f72c3893d0487ab09321552ae022ceaa89cd2499f690e8"
EDGE_STATS_DIM = 12
RELATION_TO_INDEX = {"call": 0, "trans": 1}
THRESHOLD = 0.5
BOOTSTRAP_REPETITIONS = 20_000
MINIMUM_PRACTICAL_DELTA = 0.005
ALPHA = 0.05
CODE_SHA256 = "91b0b511103e93fe2339fddabbacf86c18534e637d510676334f226be6d82437"
CONFIG = {'protocol_id': 'group23-rq1-graphsage-matched-20split-replication-v2.1', 'status': 'FROZEN AFTER ALL V1 RESULTS WERE OBSERVED AND BEFORE ANY V2/V2.1 RESULT', 'master_protocol_id': 'group23-rq1-all-six-matched-20split-replication-v2.1', 'master_protocol_sha256': '3e63674a98e052d1770851009078adf4e2ae192a47dc9c72b84c987d202aa17c', 'source_data_sha256': '8cc653bb389ea1a316f72c3893d0487ab09321552ae022ceaa89cd2499f690e8', 'model_pair': {'baseline': 'GraphSAGE-Norm', 'plus': 'GraphSAGE+'}, 'shared_configuration': {'hidden': 32, 'learning_rate': 0.01}, 'configuration_origin': 'frozen baseline winner; deliberately shared with plus so UFE is the only conceptual change', 'implementation': {'architecture': 'BatchNorm(14)-Linear node encoder; two mean-SAGEConv layers plus classifier; plus adds the matched UFE branch', 'dropout': 0.2, 'weight_decay': 0.0005, 'max_epochs': 1000, 'patience': 50, 'checkpoint_rule': 'strict validation Macro-F1 improvement (>)'}, 'matching_rule': 'Within each pair and split, baseline and plus share the fixed configuration, train/validation/test node IDs, model seed, threshold, and materialized sampled batches/views. Plus differs only by receiving the 12-dimensional label-free UFE input/branch.', 'split_seeds': [2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020], 'split_method': '20 predetermined stratified random 60/20/20 splits over the same 2763 labeled EOA', 'model_seeds': [12, 13, 14, 15, 16], 'primary_metric': 'test Macro-F1 at a fixed threshold of 0.5', 'ufe': {'dimensions': 12, 'definition': 'rows/count/sum x in/out x call/trans inside each sampled batch', 'transform': 'log1p', 'uses_labels': False, 'fanout': [100, 100], 'batch_size': 512}, 'selection_and_blinding': {'hyperparameter_tuning_in_v2_1': False, 'configurations': 'frozen from v1 validation decisions / official Meta-IFD default', 'test_materialization': 'after validation decisions/checkpoints are frozen within each split', 'threshold': 0.5, 'early_termination_based_on_results': False}, 'inference': {'test': 'two-sided Nadeau-Bengio corrected resampled t-test on 20 split-level paired deltas', 'k': 20, 'df': 19, 'test_train_ratio': '553/1657', 'alpha': 0.05, 'multiplicity': 'Holm correction across the six primary model-pair p-values', 'minimum_practical_delta': 0.005, 'success_rule': 'mean delta >= 0.005, Nadeau-Bengio 95% CI lower bound > 0, and Holm-adjusted p < 0.05'}, 'reporting_commitment': {'report_all_six_pairs': True, 'report_all_twenty_splits': True, 'report_v1_and_v2_1_separately': True, 'do_not_pool_v1_and_v2_1_post_hoc': True, 'paper_comparison': 'Use the already completed author/paper split as Table A. V2.1 is Table B replication and is not a direct numerical comparison with the paper table.', 'limitations': ['random re-splits reuse the same 2763 labeled nodes', 'not an external, temporal, or unseen-dataset holdout', 'PyG neighborhood sampling may differ across sessions, but variants within each pair reuse identical materialized inputs']}, 'code_sha256': '__CODE_SHA256__', 'change_control': 'Any modification after any V2.1 result is viewed requires a new version; v1, the unexecuted v2 package, v2.1, and any later version must remain distinguishable.'}
CONFIG["code_sha256"] = CODE_SHA256
PROTOCOL_JSON = json.dumps(CONFIG, sort_keys=True, indent=2) + "\n"
PROTOCOL_SHA256 = hashlib.sha256(PROTOCOL_JSON.encode("utf-8")).hexdigest()


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_int64(values):
    return hashlib.sha256(np.asarray(values, dtype=np.int64).tobytes()).hexdigest()


def write_csv(path, rows):
    assert rows
    with Path(path).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


try:
    import torch_geometric
    import pyg_lib
    import torch_scatter
    import torch_sparse
except ModuleNotFoundError:
    torch_version = torch.__version__.split("+")[0]
    cuda_tag = "cpu" if torch.version.cuda is None else "cu" + torch.version.cuda.replace(".", "")
    wheel_url = f"https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyg_lib", "torch_scatter", "torch_sparse", "-f", wheel_url])
    import torch_geometric
    import pyg_lib
    import torch_scatter
    import torch_sparse

from scipy.stats import binomtest, t as student_t, wilcoxon
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score, matthews_corrcoef,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader

data_candidates = [p for p in Path("/kaggle/input").rglob("data.pt")
                   if "phish/processed/data.pt" in str(p).replace("\\", "/").lower()]
assert len(data_candidates) == 1, [str(p) for p in data_candidates]
data_sha256 = sha256_file(data_candidates[0])
assert data_sha256 == SOURCE_DATA_SHA256
loaded = torch.load(data_candidates[0], map_location="cpu", weights_only=False)
data = loaded[0] if isinstance(loaded, tuple) and len(loaded) == 2 else loaded
assert isinstance(data, HeteroData)
del loaded
gc.collect()
assert set(data.node_types) == {"EOA", "CA"}
assert data["EOA"].x.shape[1] == data["CA"].x.shape[1] == 14
labeled_mask = (data["EOA"].y == 0) | (data["EOA"].y == 1)
labeled_ids = torch.where(labeled_mask)[0].cpu().numpy().astype(np.int64)
labeled_y = data["EOA"].y[labeled_mask].cpu().numpy().astype(np.int64)
assert len(labeled_ids) == 2763 and int(labeled_y.sum()) == 1206
fanout_by_relation = {edge_type: [100, 100] for edge_type in data.edge_types}

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir = Path("/kaggle/working") / f"rq1_v2_1_20split_{MODEL_KEY}_{timestamp}"
run_dir.mkdir(parents=True, exist_ok=False)
(run_dir / "protocol.json").write_text(PROTOCOL_JSON, encoding="utf-8")
assert sha256_file(run_dir / "protocol.json") == PROTOCOL_SHA256
print("Model pair:", VARIANT_LABEL)
print("Protocol SHA256:", PROTOCOL_SHA256)
print("Code SHA256:", CODE_SHA256)
print("Data SHA256:", data_sha256)
print("Torch/PyG:", torch.__version__, torch_geometric.__version__)
print("Output:", run_dir)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 45.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 93.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 100.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 84.9 MB/s eta 0:00:00
Model pair: {'base': 'GraphSAGE-Norm', 'plus': 'GraphSAGE+'}
Protocol SHA256: dff7bbb3f96f616cabdcfdaeaa1a91dbf34f6401b38695ca4d5377fd2c83e343
Code SHA256: 91b0b511103e93fe2339fddabbacf86c18534e637d510676334f226be6d82437
Data SHA256: 8cc653bb389ea1a316f72c3893d0487ab09321552ae022ceaa89cd2499f690e8
Torch/PyG: 2.10.0+cu128 2.8.0.post1
Output: /kaggle/working/rq1_v2_1_20split_graphsage_20260925T181233141222Z


## Hàm mô hình, UFE và matched split


In [2]:

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, SAGEConv

assert torch.cuda.is_available(), "Hãy bật GPU trong Kaggle"
device = torch.device("cuda:0")
torch.use_deterministic_algorithms(True, warn_only=True)


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False


def batches_fingerprint(batches):
    digest = hashlib.sha256()
    for batch in batches:
        for nt in ("EOA", "CA"):
            digest.update(batch[nt].n_id.detach().cpu().numpy().astype(np.int64, copy=False).tobytes())
        for et in sorted(batch.edge_types):
            digest.update(batch[et].edge_index.detach().cpu().numpy().astype(np.int64, copy=False).tobytes())
    return digest.hexdigest()


def materialize_split(mask, seed, expected, shuffle):
    set_seed(seed)
    loader = NeighborLoader(data, input_nodes=("EOA", mask), num_neighbors=fanout_by_relation, batch_size=512, shuffle=shuffle, num_workers=0)
    batches = [batch.cpu() for batch in loader]; ids=[]
    for batch in batches:
        n=int(batch["EOA"].batch_size); ids.extend(batch["EOA"].n_id[:n].tolist())
        assert bool(torch.all((batch["EOA"].y[:n]==0)|(batch["EOA"].y[:n]==1)))
        batch["EOA"].y[n:] = -1
        if "CA" in batch.node_types and hasattr(batch["CA"], "y"): batch["CA"].y[:] = -1
    assert len(ids)==len(set(ids))==expected and set(ids)==set(torch.where(mask)[0].tolist())
    return batches, ids


def homogenize(batch, compute_ufe=True):
    offsets, xs, offset = {}, [], 0
    for nt in ("EOA", "CA"):
        offsets[nt]=offset; xs.append(batch[nt].x); offset += batch[nt].x.shape[0]
    x=torch.cat(xs,dim=0); ufe=x.new_zeros((x.shape[0],EDGE_STATS_DIM)) if compute_ufe else None; parts=[]
    for et, local in batch.edge_index_dict.items():
        if local.numel()==0: continue
        st,rel,dt=et; remap=local.clone(); remap[0]+=offsets[st]; remap[1]+=offsets[dt]; parts.append(remap)
        if compute_ufe:
            attr=torch.nan_to_num(batch[et].edge_attr[:,:2].to(dtype=x.dtype),nan=0.0,posinf=0.0,neginf=0.0).clamp_min_(0.0)
            src,dst=remap[0],remap[1]; ones=torch.ones(src.numel(),dtype=x.dtype,device=x.device); col=RELATION_TO_INDEX[rel]*6
            ufe[:,col+0].index_add_(0,dst,ones); ufe[:,col+1].index_add_(0,src,ones)
            ufe[:,col+2].index_add_(0,dst,attr[:,0]); ufe[:,col+3].index_add_(0,src,attr[:,0])
            ufe[:,col+4].index_add_(0,dst,attr[:,1]); ufe[:,col+5].index_add_(0,src,attr[:,1])
    edge_index=torch.cat(parts,dim=1)
    if compute_ufe: ufe=torch.log1p(ufe)
    return x,edge_index,ufe


class MatchedGraph(nn.Module):
    def __init__(self, use_ufe):
        super().__init__(); hidden=SHARED_CONFIG["hidden"]; self.use_ufe=use_ufe; self.dropout=0.2; self.edge_hidden=max(16,hidden//2)
        in_dim=hidden+(self.edge_hidden if use_ufe else 0)
        self.node_norm=nn.BatchNorm1d(14); self.node_encoder=nn.Linear(14,hidden)
        if MODEL_KEY=="gat":
            self.conv1=GATConv(in_dim,hidden,heads=4,concat=True,dropout=self.dropout); self.conv2=GATConv(hidden*4,2,heads=4,concat=False,dropout=self.dropout)
        else:
            self.conv1=SAGEConv(in_dim,hidden,aggr="mean"); self.conv2=SAGEConv(hidden,hidden,aggr="mean"); self.classifier=nn.Linear(hidden,2)
        if use_ufe:
            self.ufe_norm=nn.BatchNorm1d(EDGE_STATS_DIM); self.ufe_encoder=nn.Linear(EDGE_STATS_DIM,self.edge_hidden)
    def forward(self,batch):
        x,edge_index,ufe=homogenize(batch,self.use_ufe); h=F.relu(self.node_encoder(self.node_norm(x)))
        if self.use_ufe: h=torch.cat([h,F.relu(self.ufe_encoder(self.ufe_norm(ufe)))],dim=1)
        h=F.dropout(h,p=self.dropout,training=self.training); h=self.conv1(h,edge_index); h=F.elu(h) if MODEL_KEY=="gat" else F.relu(h)
        h=F.dropout(h,p=self.dropout,training=self.training); h=self.conv2(h,edge_index)
        return self.classifier(F.relu(h)) if MODEL_KEY=="graphsage" else h


def build_model(variant,seed):
    set_seed(seed); return MatchedGraph(variant=="plus").to(device)


def metric_bundle(labels,scores,loss=None,seconds=None):
    labels=np.asarray(labels,dtype=np.int64);scores=np.asarray(scores,dtype=np.float64);pred=(scores>=THRESHOLD).astype(np.int64)
    tn,fp,fn,tp=confusion_matrix(labels,pred,labels=[0,1]).ravel()
    out={"pr_auc":float(average_precision_score(labels,scores)),"roc_auc":float(roc_auc_score(labels,scores)),
         "precision":float(precision_score(labels,pred,zero_division=0)),"recall":float(recall_score(labels,pred,zero_division=0)),
         "binary_f1":float(f1_score(labels,pred,zero_division=0)),"micro_f1":float(f1_score(labels,pred,average="micro")),
         "macro_f1":float(f1_score(labels,pred,average="macro")),"mcc":float(matthews_corrcoef(labels,pred)),"fpr":float(fp/(fp+tn)),
         "tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp)}
    if loss is not None:out["loss"]=float(loss)
    if seconds is not None:out["seconds"]=float(seconds)
    return out


def run_batches(model,batches,optimizer=None,collect_rows=False):
    training=optimizer is not None;model.train(training);labels_all=[];scores_all=[];losses=[];rows=[]
    torch.cuda.synchronize();started=time.perf_counter()
    for stored in batches:
        batch=stored.to(device);n=int(batch["EOA"].batch_size);labels=batch["EOA"].y[:n]
        if training:optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits=model(batch)[:n];loss=F.cross_entropy(logits,labels)
            if training:loss.backward();optimizer.step()
        scores=torch.softmax(logits.detach(),dim=1)[:,1]; lv=labels.detach().cpu().tolist();sv=scores.detach().cpu().tolist()
        labels_all.extend(lv);scores_all.extend(sv);losses.append(float(loss.detach().cpu())*n)
        if collect_rows:
            ids=batch["EOA"].n_id[:n].detach().cpu().tolist();rows.extend({"node_id":int(i),"label":int(y),"score":float(s),"prediction":int(s>=THRESHOLD),"threshold":THRESHOLD} for i,y,s in zip(ids,lv,sv))
    torch.cuda.synchronize();return metric_bundle(labels_all,scores_all,sum(losses)/len(labels_all),time.perf_counter()-started),rows


def clone_state(model):return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}


def train_one(variant,seed,train_batches,validation_batches,folder):
    model=build_model(variant,seed);params=sum(p.numel() for p in model.parameters());opt=torch.optim.Adam(model.parameters(),lr=SHARED_CONFIG["learning_rate"],weight_decay=5e-4)
    best=-1.0;best_epoch=0;state=None;stale=0;history=[];started=time.perf_counter()
    for epoch in range(1,1001):
        tr,_=run_batches(model,train_batches,opt);va,_=run_batches(model,validation_batches)
        if va["macro_f1"]>best:best=va["macro_f1"];best_epoch=epoch;state=clone_state(model);stale=0
        else:stale+=1
        history.append({"epoch":epoch,"train_loss":tr["loss"],"train_macro_f1":tr["macro_f1"],"validation_loss":va["loss"],"validation_macro_f1":va["macro_f1"],"validation_pr_auc":va["pr_auc"],"validation_fpr":va["fpr"],"best_epoch":best_epoch,"stale_epochs":stale})
        if stale>=50:break
    seconds=time.perf_counter()-started;model.load_state_dict(state,strict=True);va,_=run_batches(model,validation_batches);assert abs(va["macro_f1"]-best)<5e-3
    folder.mkdir(parents=True,exist_ok=True);path=folder/f"{variant}_seed_{seed}.pt";torch.save({"state_dict":state,"variant":variant,"seed":seed,"config":SHARED_CONFIG,"parameter_count":params,"protocol_sha256":PROTOCOL_SHA256,"code_sha256":CODE_SHA256,"source_data_sha256":data_sha256},path)
    write_csv(folder/f"history_{variant}_seed_{seed}.csv",history);br=history[best_epoch-1]
    row={"variant":variant,"seed":seed,"best_epoch":best_epoch,"trained_epochs":len(history),"training_seconds":seconds,"parameter_count":params,"train_macro_f1_at_best":br["train_macro_f1"],**{f"validation_{k}":v for k,v in va.items()},"checkpoint_sha256":sha256_file(path)}
    del model,opt,state;gc.collect();torch.cuda.empty_cache();return row,path


def evaluate(variant,seed,path,digest,test_batches):
    assert sha256_file(path)==digest;ck=torch.load(path,map_location="cpu",weights_only=False);model=build_model(variant,seed);model.load_state_dict(ck["state_dict"],strict=True);result,rows=run_batches(model,test_batches,collect_rows=True);del model,ck;gc.collect();torch.cuda.empty_cache();return result,rows


def run_matched_split(split_key,masks,sampler_seeds):
    folder=run_dir/split_key;folder.mkdir(parents=True,exist_ok=False)
    train_batches,train_ids=materialize_split(masks["train"],sampler_seeds["train"],1657,True);val_batches,val_ids=materialize_split(masks["validation"],sampler_seeds["validation"],553,True)
    train_hash=batches_fingerprint(train_batches);val_hash=batches_fingerprint(val_batches);rows=[];paths={}
    for variant in VARIANTS:
        for seed in MODEL_SEEDS:
            row,path=train_one(variant,seed,train_batches,val_batches,folder);rows.append(row);paths[(variant,seed)]=path
            print(split_key,VARIANT_LABEL[variant],seed,row["validation_macro_f1"],flush=True)
    write_csv(folder/"validation_runs.csv",rows)
    decision={"status":"FROZEN VALIDATION DECISION (fixed shared config)","split_key":split_key,"protocol_sha256":PROTOCOL_SHA256,"code_sha256":CODE_SHA256,"source_data_sha256":data_sha256,"shared_config":SHARED_CONFIG,"sampler_seeds":sampler_seeds,"train_target_ids_sha256":sha256_int64(sorted(train_ids)),"validation_target_ids_sha256":sha256_int64(sorted(val_ids)),"train_batches_sha256":train_hash,"validation_batches_sha256":val_hash,"checkpoint_sha256":{f"{r['variant']}_seed_{r['seed']}":r["checkpoint_sha256"] for r in rows},"test_batches_sampled_before_freeze":False,"test_metrics_computed_before_freeze":False}
    (folder/"frozen_validation_decision.json").write_text(json.dumps(decision,indent=2),encoding="utf-8")
    test_batches,test_ids=materialize_split(masks["test"],sampler_seeds["test"],553,False);test_hash=batches_fingerprint(test_batches);results=[];predictions=[];cube={}
    for variant in VARIANTS:
        scores=[]
        for seed in MODEL_SEEDS:
            result,prow=evaluate(variant,seed,paths[(variant,seed)],decision["checkpoint_sha256"][f"{variant}_seed_{seed}"],test_batches);scores.append([r["score"] for r in prow])
            for r in prow:r.update({"split_key":split_key,"variant":variant,"model":VARIANT_LABEL[variant],"seed":seed})
            predictions.extend(prow);vr=next(r for r in rows if r["variant"]==variant and r["seed"]==seed);results.append({"split_key":split_key,"variant":variant,"model":VARIANT_LABEL[variant],"seed":seed,"validation_macro_f1":vr["validation_macro_f1"],"best_epoch":vr["best_epoch"],**{f"test_{k}":v for k,v in result.items()}})
        cube[variant]=np.asarray(scores)
    labels=np.asarray([r["label"] for r in predictions if r["variant"]=="base" and r["seed"]==MODEL_SEEDS[0]])
    write_csv(folder/"test_results.csv",results);write_csv(folder/"test_predictions.csv",predictions)
    sr={"split_key":split_key,"protocol_sha256":PROTOCOL_SHA256,"code_sha256":CODE_SHA256,"source_data_sha256":data_sha256,"test_target_ids_sorted_sha256":sha256_int64(sorted(test_ids)),"test_batches_sha256":test_hash,"base_mean_macro_f1":float(np.mean([r["test_macro_f1"] for r in results if r["variant"]=="base"])),"plus_mean_macro_f1":float(np.mean([r["test_macro_f1"] for r in results if r["variant"]=="plus"])),"decision":decision};sr["delta"]=sr["plus_mean_macro_f1"]-sr["base_mean_macro_f1"]
    np.savez_compressed(folder/"score_cube.npz",base=cube["base"],plus=cube["plus"],labels=labels,test_ids=np.asarray(test_ids));sr["score_cube_sha256"]=sha256_file(folder/"score_cube.npz");(folder/"split_result.json").write_text(json.dumps(sr,indent=2),encoding="utf-8")
    del train_batches,val_batches,test_batches;gc.collect();torch.cuda.empty_cache();return sr,cube,labels


def mean_macro(score_matrix, labels, sample):
    truth = labels[sample].astype(bool)
    pred = score_matrix[:, sample] >= THRESHOLD
    tp = (pred & truth).sum(1); fp = (pred & ~truth).sum(1)
    fn = (~pred & truth).sum(1); tn = (~pred & ~truth).sum(1)
    f1_pos = np.divide(2 * tp, 2 * tp + fp + fn, out=np.zeros(len(tp)), where=(2 * tp + fp + fn) > 0)
    f1_neg = np.divide(2 * tn, 2 * tn + fp + fn, out=np.zeros(len(tn)), where=(2 * tn + fp + fn) > 0)
    return float(np.mean((f1_pos + f1_neg) / 2))


def summarize_table_a(result_a, cube_a, labels_a):
    nodes = np.arange(len(labels_a))
    observed = mean_macro(cube_a["plus"], labels_a, nodes) - mean_macro(cube_a["base"], labels_a, nodes)
    rng = np.random.default_rng(20260925)
    boot = np.empty(BOOTSTRAP_REPETITIONS)
    for i in range(BOOTSTRAP_REPETITIONS):
        sample = rng.integers(0, len(labels_a), size=len(labels_a))
        boot[i] = mean_macro(cube_a["plus"], labels_a, sample) - mean_macro(cube_a["base"], labels_a, sample)
    ci = np.quantile(boot, [0.025, 0.975])
    p = min(1.0, 2 * min((np.sum(boot <= 0) + 1) / (BOOTSTRAP_REPETITIONS + 1),
                         (np.sum(boot >= 0) + 1) / (BOOTSTRAP_REPETITIONS + 1)))
    per_seed = []
    for k, seed in enumerate(MODEL_SEEDS):
        plus_ok = (cube_a["plus"][k] >= THRESHOLD) == labels_a
        base_ok = (cube_a["base"][k] >= THRESHOLD) == labels_a
        only_plus = int(np.sum(plus_ok & ~base_ok)); only_base = int(np.sum(~plus_ok & base_ok))
        per_seed.append({"seed": seed,
                         "plus_macro_f1": mean_macro(cube_a["plus"][[k]], labels_a, nodes),
                         "base_macro_f1": mean_macro(cube_a["base"][[k]], labels_a, nodes),
                         "only_plus_correct": only_plus, "only_base_correct": only_base,
                         "mcnemar_p": 1.0 if only_plus + only_base == 0 else float(binomtest(min(only_plus, only_base), only_plus + only_base, 0.5).pvalue)})
    return {"role": "retrospective matched comparison on the previously viewed paper split",
            "base_mean_macro_f1": result_a["base_mean_macro_f1"], "plus_mean_macro_f1": result_a["plus_mean_macro_f1"],
            "delta": observed, "bootstrap_95ci": [float(ci[0]), float(ci[1])], "bootstrap_p_two_sided": float(p),
            "seeds_plus_better": int(sum(r["plus_macro_f1"] > r["base_macro_f1"] for r in per_seed)), "per_seed": per_seed}


def summarize_table_b(split_results):
    deltas = np.asarray([r["delta"] for r in split_results])
    k = len(deltas); assert k == 20
    ratio = EXPECTED_COUNTS["test"] / EXPECTED_COUNTS["train"]
    variance = float(deltas.var(ddof=1))
    se = float(np.sqrt((1 / k + ratio) * variance))
    mean_delta = float(deltas.mean())
    if se == 0:
        t_stat, p, ci = (float("inf") if mean_delta else 0.0), (0.0 if mean_delta else 1.0), (mean_delta, mean_delta)
    else:
        t_stat = mean_delta / se
        p = float(2 * student_t.sf(abs(t_stat), df=k - 1))
        crit = float(student_t.ppf(0.975, df=k - 1))
        ci = (mean_delta - crit * se, mean_delta + crit * se)
    try:
        wp = float(wilcoxon(deltas).pvalue)
    except ValueError:
        wp = 1.0
    return {
        "role": "fixed-budget v2.1 20-split replication; same nodes, not an external holdout",
        "per_split": [{"split_key": r["split_key"], "base": r["base_mean_macro_f1"], "plus": r["plus_mean_macro_f1"], "delta": r["delta"]} for r in split_results],
        "base_mean_over_splits": float(np.mean([r["base_mean_macro_f1"] for r in split_results])),
        "plus_mean_over_splits": float(np.mean([r["plus_mean_macro_f1"] for r in split_results])),
        "mean_delta": mean_delta, "sd_delta": float(np.sqrt(variance)),
        "nadeau_bengio": {"se": se, "t": t_stat, "df": k - 1, "p_two_sided_raw": p, "ci95": [ci[0], ci[1]], "ratio_test_train": ratio},
        "descriptive_wilcoxon_p": wp, "splits_plus_better": int(np.sum(deltas > 0)), "splits_tied": int(np.sum(deltas == 0)),
        "raw_pair_rule_met": bool(mean_delta >= MINIMUM_PRACTICAL_DELTA and ci[0] > 0 and p < ALPHA),
        "global_holm_pending": True,
    }


## Chạy đủ 20 split v2.1


In [3]:

split_results=[]
for split_seed in RESPLIT_SEEDS:
    pos=np.arange(len(labeled_ids));trainval_pos,test_pos=train_test_split(pos,test_size=.20,random_state=split_seed,stratify=labeled_y)
    train_pos,val_pos=train_test_split(trainval_pos,test_size=.25,random_state=split_seed,stratify=labeled_y[trainval_pos])
    assert (len(train_pos),len(val_pos),len(test_pos)) == (1657,553,553)
    masks={}
    for name,p in (("train",train_pos),("validation",val_pos),("test",test_pos)):
        m=torch.zeros(data["EOA"].num_nodes,dtype=torch.bool);m[torch.from_numpy(labeled_ids[p])]=True;masks[name]=m
    assert not bool((masks["train"] & masks["validation"]).any() or (masks["train"] & masks["test"]).any() or (masks["validation"] & masks["test"]).any())
    seeds={"train":split_seed*10+1,"validation":split_seed*10+2,"test":split_seed*10+3}
    result,_,_=run_matched_split(f"V2_split_{split_seed}",masks,seeds);split_results.append(result)
    print(result["split_key"],result["base_mean_macro_f1"],result["plus_mean_macro_f1"],result["delta"],flush=True)

table_b=summarize_table_b(split_results)
write_csv(run_dir/"v2_20split_per_split.csv",table_b["per_split"])
final={"status":"completed fixed-budget v2.1 matched pair on all 20 predetermined splits","model_key":MODEL_KEY,"model_label":MODEL_LABEL,"variant_labels":VARIANT_LABEL,"master_protocol_sha256":CONFIG["master_protocol_sha256"],"protocol_sha256":PROTOCOL_SHA256,"code_sha256":CODE_SHA256,"protocol":CONFIG,"source_data_sha256":data_sha256,"v2_20split":table_b,"all_20_splits_completed":len(split_results)==20,"post_test_selection":False,"protocol_changed_after_results":False,"paper_comparison_in_this_notebook":False}
(run_dir/"final_summary.json").write_text(json.dumps(final,indent=2),encoding="utf-8")
shutil.make_archive(str(run_dir),"zip",run_dir)
print("V2 20-split mean:",table_b["base_mean_over_splits"],"->",table_b["plus_mean_over_splits"],"delta",table_b["mean_delta"])
print("NB:",table_b["nadeau_bengio"])
print("Raw pair rule met:",table_b["raw_pair_rule_met"],"| final Holm pending across all six pairs")
print("Output:",run_dir,"ZIP:",str(run_dir)+".zip")


V2_split_2001 GraphSAGE-Norm 12 0.9504258180188256
V2_split_2001 GraphSAGE-Norm 13 0.9560657821561929
V2_split_2001 GraphSAGE-Norm 14 0.9543236430074702
V2_split_2001 GraphSAGE-Norm 15 0.9597897937599154
V2_split_2001 GraphSAGE-Norm 16 0.9489327458378407
V2_split_2001 GraphSAGE+ 12 0.9816474180273463
V2_split_2001 GraphSAGE+ 13 0.9798210666339363
V2_split_2001 GraphSAGE+ 14 0.9816634834739244
V2_split_2001 GraphSAGE+ 15 0.9816474180273463
V2_split_2001 GraphSAGE+ 16 0.9779769016328156
V2_split_2001 0.9414563318186762 0.978697003496368 0.03724067167769185
V2_split_2002 GraphSAGE-Norm 12 0.9450822276952411
V2_split_2002 GraphSAGE-Norm 13 0.9397072087698606
V2_split_2002 GraphSAGE-Norm 14 0.9446202531645569
V2_split_2002 GraphSAGE-Norm 15 0.9469754831456959
V2_split_2002 GraphSAGE-Norm 16 0.9414675451827155
V2_split_2002 GraphSAGE+ 12 0.9871234927234928
V2_split_2002 GraphSAGE+ 13 0.9871234927234928
V2_split_2002 GraphSAGE+ 14 0.987135645660316
V2_split_2002 GraphSAGE+ 15 0.98712349272349